
# ML-QSPR Pipeline  
## Repeated External Validation + Visualization

This notebook implements a machine learning-assisted QSPR workflow including:

- RDKit molecular descriptor generation
- Hybrid topological + chemical descriptor modeling
- Linear Regression
- ElasticNet Regression
- Partial Least Squares (PLS)
- Repeated external validation
- Observed vs Predicted visualization
- External R² distribution boxplots

The notebook is designed for advanced QSPR and cheminformatics studies.


In [ ]:
# ==========================================================
# ML QSPR PIPELINE
# Repeated External Validation + Visualization
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.metrics import r2_score

# ==========================================================
# 1. LOAD DATA
# ==========================================================
base_path = r"D:\Phd\PhD\Papers on topological indices\30 kinas inhibitor"

df = pd.read_excel(
    base_path + r"\complete data.xlsx",
    sheet_name="Indices"
)

# ==========================================================
# 2. RDKit DESCRIPTORS
# ==========================================================
def rdkit_features(smiles):

    mol = Chem.MolFromSmiles(smiles)

    atoms = [a.GetSymbol() for a in mol.GetAtoms()]

    return {

        "NumN": atoms.count("N"),
        "NumO": atoms.count("O"),
        "NumS": atoms.count("S"),

        "NumHeteroatoms": Descriptors.NumHeteroatoms(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "MolLogP": Descriptors.MolLogP(mol),

        "RingCount": Descriptors.RingCount(mol),

        "AromaticRingCount":
            rdMolDescriptors.CalcNumAromaticRings(mol),

        "AliphaticRingCount":
            rdMolDescriptors.CalcNumAliphaticRings(mol),
    }

df = pd.concat(
    [
        df,
        df["SMILES"].apply(rdkit_features).apply(pd.Series)
    ],
    axis=1
)

# ==========================================================
# 3. SETUP
# ==========================================================
properties = ["TPSA", "HBA", "RB"]

topo = [
    "M1", "M2", "Sombor", "Mostar", "Szeged",
    "PI", "GA", "Randic", "Energy",
    "Laplacian_Energy"
]

chem = [
    "NumN", "NumO", "NumS", "NumHeteroatoms",
    "HeavyAtomCount", "FractionCSP3",
    "MolLogP", "RingCount",
    "AromaticRingCount", "AliphaticRingCount"
]

models = {

    "Linear": (
        LinearRegression(),
        False
    ),

    "ElasticNet": (
        ElasticNetCV(cv=5),
        True
    ),

    "PLS": (
        PLSRegression(n_components=3),
        True
    )
}

# ==========================================================
# 4. MAIN LOOP
# ==========================================================
for prop in properties:

    print("\n================================================")
    print(f"Target Property : {prop}")
    print("================================================")

    y = df[prop].values

    # ------------------------------------------------------
    # Best Topological Descriptor Selection
    # ------------------------------------------------------
    best = df[topo].corrwith(df[prop]).abs().idxmax()

    print(f"Best Descriptor : {best}")

    X = df[[best] + chem].values

    ext_scores = {}

    # ------------------------------------------------------
    # Model Training
    # ------------------------------------------------------
    for name, (model, scale) in models.items():

        print(f"\nRunning Model : {name}")

        r2_list = []

        for seed in range(5):

            X_tr, X_te, y_tr, y_te = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=seed
            )

            # --------------------------------------------------
            # Feature Scaling
            # --------------------------------------------------
            if scale:

                sc = StandardScaler()

                X_tr = sc.fit_transform(X_tr)
                X_te = sc.transform(X_te)

            # --------------------------------------------------
            # Model Fitting
            # --------------------------------------------------
            model.fit(X_tr, y_tr)

            y_pred = model.predict(X_te)

            r2 = r2_score(y_te, y_pred)

            r2_list.append(r2)

            print(f"Seed {seed}  --->  External R² = {r2:.4f}")

            # --------------------------------------------------
            # Observed vs Predicted Plot
            # --------------------------------------------------
            if seed == 0:

                plt.figure()

                plt.scatter(y_te, y_pred)

                plt.plot(
                    [y.min(), y.max()],
                    [y.min(), y.max()],
                    '--'
                )

                plt.xlabel("Observed")
                plt.ylabel("Predicted")

                plt.title(
                    f"{prop} ({name}) External Prediction"
                )

                plt.savefig(
                    base_path +
                    fr"\{prop}_{name}_Obs_vs_Pred.png",
                    dpi=300,
                    bbox_inches='tight'
                )

                plt.close()

        ext_scores[name] = r2_list

    # ======================================================
    # Boxplot of External Validation
    # ======================================================
    plt.figure()

    plt.boxplot(
        ext_scores.values(),
        labels=ext_scores.keys()
    )

    plt.ylabel("External R²")

    plt.title(
        f"{prop}: External R² Distribution"
    )

    plt.savefig(
        base_path +
        fr"\{prop}_External_R2_Boxplot.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

print("\n✔ ML QSPR + Visuals Completed")



## Generated Outputs

The notebook automatically generates:

- External validation R² values
- Observed vs Predicted plots
- External validation boxplots
- Descriptor selection reports
- Machine learning model performance summaries

### Implemented Models
- Linear Regression
- ElasticNet Regression
- Partial Least Squares (PLS)

### Important Note
Before running the notebook, update the `base_path` according to your local system directory.
